# ADVENTUREWORKS SALES ANALYSIS

## EXECUTIVE SUMMARY

This report analyzes sales performance for the dataset AdventureWorks using SQL,Python and Plotly. The analysis focuses on 3 main business perspectives:
1. Revenue Analysis - 
    This aims to evaluate overall sales performance of the company and identify the product and category wise trend of revenue generation.
2. Product Performance Analysis - 
    This aims to identifies high performing products and categories. Focus on possible contributing factors in sales volume and revenue generation. Perform a multi dimension analysis to visualize the relationship between these factos.
3. Order Analysis - 
    This examines the geographical distribution of sales order value across countries and cities.

The insights generated after each section can support business decisions related to product management, regional marketing, inventory allocation and business growth.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sqlite3 as sql4
from plotly.subplots import make_subplots

In [3]:
conn= sql4.connect("data/AdventureWorks-sqlite.db")

In [4]:
def run_query(query,conn):
    df=pd.read_sql(query,conn)
    return df

## ---REVENUE ANALYSIS

## BUSINESS QUESTION
The objective of this analysis is to identify the primary sources of revenue within the business. Specifically we aim to determine:

1. Which product categories generate the highest revenue?
2. Which individual products contribute the most to total revenue?
3. How concentrated is revenue across categories and products?

## BUSINESS CONTEXT
Understanding revenue composition can help a business distinguish its most valuable products and categories. This insight can support decision making in context of inventory planning, pricing, project feasibility, product promotion, R&D etc.

SQL Query 

---- 

Join relevant tables to work out product and its hierarchy.
Query the revenue by product category


In [5]:
query="""
WITH ch AS
(
SELECT ProductCategoryID AS ParentProductCategoryID, Name AS CategoryName
FROM ProductCategory
WHERE ParentProductCategoryID = ""
)
SELECT SUM(sod.LineTotal) AS RevenueByCategory,ch.CategoryName
FROM ProductCategory pcid
JOIN ch
ON pcid.ParentProductCategoryID = ch.ParentProductCategoryID
JOIN Product p
ON p.ProductCategoryID=pcid.ProductCategoryID
JOIN SalesOrderDetail sod
ON sod.ProductID = p.ProductID
GROUP BY ch.CategoryName
ORDER BY RevenueByCategory DESC

"""

In [6]:
df=run_query(query,conn)
df.head()
sum_revenuebycat=df["RevenueByCategory"].sum()
print(f"The sum of all revenue from the dataframe is {sum_revenuebycat}")

The sum of all revenue from the dataframe is 708690.1530579994


## VALIDATION
The calculated revenue was validated by cross checking it with the sum taken directly from the SalesOrderDetail Table. The values match indicating the joins and aggregation was performed correctly.

In [7]:
query = """    

SELECT SUM(LineTotal)
FROM SalesOrderDetail
"""

validation=run_query(query,conn)
validation.head()

,SUM(LineTotal)
0,708690.153058


## VISUALIZATION RATIONALE

A simple bar chart is used to identify the top performing categories and also the overall sales performance by the business


In [8]:
fig=go.Figure()

fig.add_trace(
    go.Bar(
        x=df["RevenueByCategory"],
        y=df["CategoryName"],
        orientation="h"
    )
        


    )
fig.update_layout(
    title="Revenue By Category",
    xaxis_title="Revenue Generated ($)"
)

fig.show()

## INSIGHTS
Bike products generated the most revenue while accessories contributed the least. 


## SQL QUERY
----
Join relevant tables to work out product and its hierarchy.
Query the revenue by individual products (sub categories)

Restricting our result to the top 10 as the number of individual product variant is large at dataframe level, alternatively it could be done at query level using Rank()

In [9]:
query="""


SELECT SUM(sod.LineTotal) AS RevenueByProduct, pcid.Name AS ProductName
FROM ProductCategory pcid

JOIN Product p
ON p.ProductCategoryID=pcid.ProductCategoryID

JOIN SalesOrderDetail sod
ON sod.ProductID=p.ProductID

GROUP BY pcid.ProductCategoryID, pcid.Name


"""

df=run_query(query,conn).sort_values(by="RevenueByProduct",ascending=True)


## VISUALIZATION RATIONALE

A simple bar chart is used to understand the proportion of revenue generation by different product subcategories and to identify any exceptionally high performing product. 

In [10]:
fig=go.Figure()

fig.add_trace(
    go.Bar(
        x=df["RevenueByProduct"],
        y=df["ProductName"].tail(10),
        orientation="h"
  
        
    )
        


    )
fig.update_layout(
    title="Top 10 Revenue By Product",
    xaxis_title="Revenue Generated ($)"
)

fig.show()

## INSIGHT 

Touring Bikes take lead as the highest revenue generating product followed by 2 other bike product variants.
Followed next by category of 'Components' with frames variant making the most revenue in that category.

## SQL QUERY
----
Join relevant tables to work out product and its hierarchy.
Query the revenue by individual products (sub categories)

Restricting our result to the top 10 as the number of individual product variant is large at dataframe level, alternatively it could be done at query level using Rank()

## VISUALIZATION RATIONALE

--- A Pareto chart will make the observation more refined as we see numerically how products distribute revenue in 80/20 portions.

In [11]:
df=df.sort_values(by="RevenueByProduct",ascending=False)

# 2. Calculate cumulative percentage
df["CumulativeSum"] = df["RevenueByProduct"].cumsum()
total_revenue = df["RevenueByProduct"].sum()
df["CumulativePercentage"] = (df["CumulativeSum"] / total_revenue) * 100



fig = make_subplots(specs=[[{"secondary_y": True}]])

# 4. Add bars for individual revenue
fig.add_trace(
    go.Bar(
        x=df["ProductName"],
        y=df["RevenueByProduct"],
        name="Revenue",
        marker_color="crimson",
    ),
    secondary_y=False,
)

# 5. Add line for cumulative percentage
fig.add_trace(
    go.Scatter(
        x=df["ProductName"],
        y=df["CumulativePercentage"],
        name="Cumulative %",
        mode="lines+markers",
        marker_color="black",
    ),
    secondary_y=True,
)

# 6. Configure layout titles and axes ranges
fig.update_layout(
    title_text="Revenue Pareto Chart",
    xaxis_title="Product",
    yaxis_title="Revenue ($)",
    yaxis2_title="Cumulative Percentage (%)",
)

# Set secondary y-axis range from 0 to 100
fig.update_yaxes(range=[0, 100], secondary_y=True)

fig.show()

# 7. Render ch

## INSIGHT
The top 3 bike variant alone make up more than 80% of the revenue generated by the company.

## COMPARISON 
The category level analysis provide a general business performance overview where as product level analysis highlight which products are driving the most revenue and performing well. Together these analysis reveal both business strategic trend and operation drivers.

## SECTION SUMMARY (REVENUE ANALYSIS)
Revenue generation is heavily concentrated within the "Bike" Category. A handful of premium products are driving the most revenue. Category Level analysis gives a general overview of the business performance

## ---PRODUCT PERFORMANCE ANALYSIS

## BUSINESS QUESTION
The objective of this analysis is to identify which factors may be affecting the performance of the products in context of revenue and sales volume & whether we can deduce any relationship between these factors. We will specifically look at pricing and profit margin as factors.

## BUSINESS CONTEXT
Understanding the relationship between pricing, sales volume, revenue and profit margin will help the business make smarter choices in terms of pricing strategy, inventory management, marketing, promotions, cost reduction etc.

## SQL QUERY

In [12]:
query="""


SELECT SUM(sod.LineTotal) AS RevneueByProduct, p.Name AS ProductName,(p.ListPrice- p.StandardCost)/p.ListPrice*100 AS ProfitMargin
FROM ProductCategory pcid

JOIN Product p
ON p.ProductCategoryID=pcid.ProductCategoryID

JOIN SalesOrderDetail sod
ON sod.ProductID=p.ProductID

GROUP BY pcid.ProductCategoryID, p.Name






"""

df=run_query(query,conn)



## VALIDATION
The unique number of products in the analysis were validated by cross checking with the total number of unique products queried directly from the SalesOrderDetail Table ensuring all of these were taken into the analysis.

In [13]:
print("The number of unique products names in this query is :")
print(int(df["ProductName"].value_counts().sum()))


query="""    

SELECT COUNT(DISTINCT(ProductID))
FROM SalesOrderDetail

"""

validation=run_query(query,conn)
print(validation.head())

The number of unique products names in this query is :
142
   COUNT(DISTINCT(ProductID))
0                         142


## VISUALIZATION RATIONALE

A scatter plot is used to visualize the relationship (if any) between revenue and profit margin across different products. 4 quadrants are setup across the median from both variables. 

A positive relationship between the two would suggest that the high performing products are relatively more profitable explaining their tendency to generate high reveneue.


In [14]:
fig=go.Figure()

fig.add_trace(


  go.Scatter(
    y=df["ProfitMargin"],
    x=df["RevneueByProduct"],
    mode='markers',
    text=df["ProductName"], # Hover text labels

))

fig.update_layout(
    title='Revenue VS Profit Margin By Product',
    template='plotly_white',
    xaxis_title="Revenue ($)",
    yaxis_title="Profit Margin %"

)


median_revenue=df["RevneueByProduct"].median()
median_margin=df["ProfitMargin"].median()

fig.add_vline(
    x=median_revenue,
    line_dash="dash",
    line_color="gray"
)
fig.add_hline(
    y=median_margin,
    line_dash="dash",
    line_color="gray"
)



fig.show()


## INSIGHT
We find no significant relationship between Revenue and Profit Margin across the different product variants. However, there are a few things that we can highlight from the analysis thus far:

-The profit margin for most product variants sit between roughly 30% - 66% which highlights a rather uniform pricing model by the business.

-High revenue generating product variants do not show high profit margin tendency rather they tend to be at slightly lower than median profit margin range.  

## SQL QUERY

In [15]:
query="""  


SELECT SUM(sod.OrderQty) AS UnitsSold,p.Name AS ProductName, (p.ListPrice- p.StandardCost)/p.ListPrice*100 AS ProfitMargin
FROM SalesOrderDetail sod

JOIN Product p
ON p.ProductID=sod.ProductID

GROUP BY p.Name




"""

df=run_query(query,conn)



## VISUALIZATION RATIONALE

A scatter plot is used to visualize the relationship (if any) between Sales Volume and profit margin across different products. 4 quadrants are setup across the median from both variables.

A positive relationship would show that products selling more are also more profitable. This would be a positive insight for the business and help the business make informed decision about these products in the future.

A negative relationship would show that products with lower profit margin tend to sell more and would allude to the common strategy with fast moving products.

In [16]:
fig=go.Figure()

fig.add_trace(


  go.Scatter(
    y=df["ProfitMargin"],
    x=df["UnitsSold"],
    mode='markers',
    text=df["ProductName"], # Hover text labels

))

fig.update_layout(
    title='Sales Volume VS Profit Margin By Product',
    template='plotly_white',
    xaxis_title="Units Sold",
    yaxis_title="Profit Margin %"

)


median_salesvolume=df["UnitsSold"].median()
median_margin=df["ProfitMargin"].median()

fig.add_vline(
    x=median_salesvolume,
    line_dash="dash",
    line_color="gray"
)
fig.add_hline(
    y=median_margin,
    line_dash="dash",
    line_color="gray"
)



fig.show()


## INSIGHT
We find no significant relationship between Sales Volume and Profit Margin across the different product variants. However, there are a few things that we can highlight from the analysis thus far:

-There are a small number of premium product that have lower sale volume and sit on the relative lower profit margin % scale but we have already deduced that they are high performing products which would allude to their absolute profit values being substantially high. Further in our analysis we will employ a bubble graph comparing all three which will zoom in on this further.

-There is a range of products with high sales volume and high profit margin. But it is premature to label them as good performers before comparing their revenues as well.

## SQL QUERY

In [17]:
query="""


SELECT SUM(sod.OrderQty) AS UnitsSold,p.Name AS ProductName, p.ListPrice AS Price, (p.ListPrice-p.StandardCost)/p.ListPrice*100 AS ProfitMargin, SUM(sod.LineTotal) AS RevenueByProduct
FROM SalesOrderDetail sod

JOIN Product p
ON p.ProductID=sod.ProductID

GROUP BY p.Name



"""
df=run_query(query,conn)

df.head(10)



,UnitsSold,ProductName,Price,ProfitMargin,RevenueByProduct
0,52,AWC Logo Cap,8.99,23.000000,277.363076
1,55,Bike Wash - Dissolver,7.95,62.600000,251.875875
2,8,Chain,20.24,55.599802,97.152000
3,34,"Classic Vest, M",63.50,62.600000,1295.400000
4,87,"Classic Vest, S",63.50,62.600000,3014.503750
5,12,Front Brakes,106.50,55.600000,766.800000
6,13,Front Derailleur,91.49,55.599956,713.622000
7,15,HL Bottom Bracket,121.49,55.599967,1093.410000
8,12,HL Crankset,404.99,55.599990,2915.928000
9,7,"HL Mountain Frame - Black, 38",1349.60,45.239997,5668.320000


## VISUALIZATION RATIONALE

A scatter plot is used to visualize the relationship (if any) between Sales Volume and pricing across different products. 4 quadrants are setup across the median from both variables.

A negative relationship would show that products selling more tend to do so because of their lower pricing. 



In [18]:
fig=go.Figure()

fig.add_trace(


  go.Scatter(
    y=df["Price"],
    x=df["UnitsSold"],
    mode='markers',
    text=df["ProductName"], # Hover text labels

))

fig.update_layout(
    title='Sales Volume VS Pricing By Product',
    template='plotly_white',
    xaxis_title="Units Sold",
    yaxis_title="Price"

)


median_salesvolume=df["UnitsSold"].median()
median_price=df["Price"].median()

fig.add_vline(
    x=median_salesvolume,
    line_dash="dash",
    line_color="gray"
)
fig.add_hline(
    y=median_price,
    line_dash="dash",
    line_color="gray"
)



fig.show()

## INSIGHT
We find no significant relationship between Sales Volume and Pricing across the different product variants. However, there are a few things that we can highlight from the analysis thus far:

There are a small number of product which tend to show that their sales volume may be a factor of lower pricing but it is premature to factualize it because the trend does not translate for the entire product range.


## SQL QUERY

In [19]:
query = """

WITH ch AS
(
SELECT ProductCategoryID AS ParentProductCategoryID, Name AS CategoryName
FROM ProductCategory
WHERE ParentProductCategoryID = ""



)


SELECT ch.CategoryName, p.Name AS ProductName, SUM(sod.OrderQty) AS UnitsSold, p.ListPrice AS Price, (p.ListPrice-p.StandardCost)/p.ListPrice*100 AS ProfitMargin, SUM(sod.LineTotal) AS RevenueByProduct
FROM ch
LEFT JOIN ProductCategory pcid
ON pcid.ParentProductCategoryID= ch.ParentProductCategoryID




JOIN Product p
ON p.ProductCategoryID=pcid.ProductCategoryID

JOIN SalesOrderDetail sod
ON sod.ProductID = p.ProductID



GROUP BY ch.CategoryName,p.Name




"""


df=run_query(query,conn)

df.head(10)




,CategoryName,ProductName,UnitsSold,Price,ProfitMargin,RevenueByProduct
0,Clothing,AWC Logo Cap,52,8.99,23.000000,277.363076
1,Accessories,Bike Wash - Dissolver,55,7.95,62.600000,251.875875
2,Components,Chain,8,20.24,55.599802,97.152000
3,Clothing,"Classic Vest, M",34,63.50,62.600000,1295.400000
4,Clothing,"Classic Vest, S",87,63.50,62.600000,3014.503750
5,Components,Front Brakes,12,106.50,55.600000,766.800000
6,Components,Front Derailleur,13,91.49,55.599956,713.622000
7,Components,HL Bottom Bracket,15,121.49,55.599967,1093.410000
8,Components,HL Crankset,12,404.99,55.599990,2915.928000
9,Components,"HL Mountain Frame - Black, 38",7,1349.60,45.239997,5668.320000


## VISUALIZATION RATIONALE

A bubble plot will help in visualizing all 4 mentioned variables and spot any relationship which may exist between them. It is a nice, clean way for visualizing the summary of analysis discussed previously. Furthermore, as discussed earlier that there were some products with relatively low profit margin % and low sales volume but ended up as high performing premium product; this visual will highlight those clearly

In [20]:

fig=px.scatter(
    df,
    x="UnitsSold",
    y="Price",
    size=np.sqrt(df["RevenueByProduct"]),
    color="ProfitMargin",
    symbol="CategoryName",
    hover_name="ProductName",
    title="Product Analysis Sales Volume VS Price (Bubble size = Revenue by Product)",
    labels={
        "UnitsSold":"Units Sold",
        "Price":"List Price ($)",
        "ProfitMargin": "Profit Margin (%)"
    }
)

fig.update_layout(
    legend=dict(
        x=1.15,
        y=-0.1,
        xanchor="left"
)
)

fig.show()

## INSIGHT
1. The pricing of premium high performing products varies within a relatively wide range while showing no significant relationship with units sold, this suggest that pricing is not a significant factor in terms of revenue generated or volume sold.
2. Category "Bike" represented with X can be observed as having generally high revenue generating products.
One variant of category "Bike" i.e Road-250-W Yellow has sold more units than other variants in category but revenue for this particular variant is not remarkably higher in comparison.
3. It is much clearer in this bubble chart that some products have high sales volume with high profit margin % but don't end up generating the highest revenue possibly due to the lower price range.

## SECTION SUMMARY 

There is no obvious relationship that can be deduced between profit margin %, sales volume, pricing and revenue generated.
This analysis has consolidated so far on our previous findings that a hand full of premium products generally associated with one category "Bike" drive the most revenue. 
There is also a range of product with high sales volume though they do not produce high revenue . This can help the business make decision such as future promotions, marketing etc related to these products.

## ---ORDER ANALYSIS

## BUSINESS QUESTION
This analysis aims to answer the following:
    1.  Which countries are represented in the dataset?
    2.  What is the geographical distribution of the sales and identify which cities,countries contribute in the order value the most?



## BUSINESS CONTEXT
Understanding where the customers generate the highest order value will help business make informed decisions. Geographical distribution of sales order value will help in making decisions about resource allocation, inventory management, marketing campaign, possible expansion etc.

Focusing on city wise distribution will help identify high-performing markets which will help in decisions about promotional activities etc.

Although a geographical global map was considered for the analysis but the dataset contains data from two cities. As a result, city wise comparision is utilized to provide a clear, more meaningful representation of the data.

## VISUALIZATION RATIONALE
A bar chart is used to quickly compare order values across cities. An interactive feature of plotly chart is employed allowing user to toggle between countries via legend, eliminating the need for separate charts country wise. 

In [21]:
query="""

SELECT  SUM(soh.SubTotal) AS OrderValue,a.CountryRegion,a.City
FROM SalesOrderHeader soh

JOIN Address a
ON a.AddressID=soh.BillToAddressID

GROUP BY a.CountryRegion,a.City
ORDER BY OrderValue DESC
"""
df=run_query(query,conn)
df.head()



,OrderValue,CountryRegion,City
0,187091.5535,United Kingdom,London
1,108561.8317,United Kingdom,Woolston
2,83858.4261,United States,Union City
3,78029.6898,United Kingdom,Liverpool
4,74058.8078,United States,Fullerton


In [22]:
fig = px.bar(
    df, 
    x='City', 
    y='OrderValue', 
    color='CountryRegion', 
    barmode='group',
    title='Order Value by Location',
    )
fig.show()

## INSIGHT
Order value is concentrated in relatively small number of cities. London records the highest value overall. United States exhibits a long-tail distribution with a few cities contributing the most in order value.

# FINAL CONCLUSIONS
## KEY FINDINGS:

Bike products are primary contributors to company revenue.

Revenue is concentrated among a relatively small number of premium products.

Profit margin alone does not explain product demand.

High sales volume do not necessarily translate into high revenue.

The dataset is limited in customer and temporal scope; therefore, findings should be interpreted within those contraints.